# Caminho dos Direitos — servidor do WhatsApp no Colab
Liga o app ao WhatsApp de verdade, seguindo o *Manual: integrando um sistema ao WhatsApp (Cloud API da Meta)*:

| Manual | Onde está |
|---|---|
| Passo 2 — webhook GET (verificação) e POST (eventos) | `GET/POST /webhook` em `servidor_whatsapp.py` |
| Validar `X-Hub-Signature-256` sobre o corpo cru | `Bot.assinatura_ok` |
| Não duplicar (`wamid` único), gravar antes de responder, 5xx se o banco falhar | `Bot.receber` + SQLite |
| Trabalho pesado fora da requisição | `BackgroundTasks` → `Bot.processar` |
| Áudio, imagem, documento: responder em vez de ignorar | "Ainda não recebo arquivos…" |
| Passo 3 — enviar, guardar o id, falha é falha | `Bot._post` (tabela `envio`) |
| Status fora de ordem | tabela `status_pendente` |
| Passo 4 — janela de 24 h e mensagem-modelo | `enviar_lembrete` (modelo) / `enviar_lembrete_na_janela` |
| Opt-in, PARAR / VOLTAR | tabela `contato` |
| LGPD: telefone mascarado no log, guardado como hash, conversa apagada em 30 dias, limite por pessoa | `mascarar`, `hash_tel`, `apagar_antigas`, `LIMITE_PROATIVO_DIA` |

**Fluxo do lembrete:** o agente salva o encaminhamento → a pessoa recebe o **texto** (mensagem-modelo) com o botão **Ouvir em áudio** → tocou no botão (ou mandou OUVIR) → recebe a **mensagem de voz** com voz neural.

**Antes de rodar:** `servidor_whatsapp.py` e `voz.py` na pasta `ia/` do Drive (a mesma do `anotacoes.py`).

Onde pegar cada segredo:

- `WA_ACCESS_TOKEN` e `WA_PHONE_ID`: https://developers.facebook.com/products/whatsapp/onboard/direct/
- `WA_APP_SECRET`: painel do app na Meta → Configurações → Básico
- `WA_VERIFY_TOKEN`: texto que vocês inventam (o mesmo vai no painel da Meta)

## 1. Drive e instalação

In [49]:
from google.colab import drive
drive.mount('/content/drive')

import sys, importlib
from pathlib import Path
PROJETO = Path('/content/drive/MyDrive/Dados/notebook_caminho_dos_direitos')   # <- mesma pasta do notebook principal
achado = sorted(PROJETO.rglob('servidor_whatsapp.py'))
assert achado, f'servidor_whatsapp.py não encontrado em {PROJETO}'
PASTA_IA = achado[0].parent
sys.path.insert(0, str(PASTA_IA))
print('Módulos em:', PASTA_IA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Módulos em: /content/drive/MyDrive/Dados/notebook_caminho_dos_direitos


In [50]:
!pip install -q fastapi uvicorn httpx edge-tts gTTS

## 2. Segredos (ícone de chave, à esquerda)
Crie no Secrets e ative para este notebook (manual, Passo 1):

| Nome | O que é |
|---|---|
| `WA_VERIFY_TOKEN` | texto longo que **vocês inventam** (o mesmo vai no painel da Meta) |
| `WA_APP_SECRET` | App Secret (painel do app → Configurações → Básico) |
| `WA_ACCESS_TOKEN` | token de acesso (o temporário dura poucas horas) |
| `WA_PHONE_ID` | ID do número de telefone de teste |

Sem `WA_ACCESS_TOKEN`/`WA_PHONE_ID`, tudo roda em **modo demonstração** (nada sai para a Meta).

In [51]:
import os
from google.colab import userdata
for nome in ['WA_VERIFY_TOKEN', 'WA_APP_SECRET', 'WA_ACCESS_TOKEN', 'WA_PHONE_ID']:
    try:
        os.environ[nome] = userdata.get(nome)
        print(f'✓ {nome}')
    except Exception:
        print(f'✗ {nome} (não encontrado)')

import voz, servidor_whatsapp as sw
importlib.reload(voz); importlib.reload(sw)

✓ WA_VERIFY_TOKEN
✓ WA_APP_SECRET
✓ WA_ACCESS_TOKEN
✓ WA_PHONE_ID


<module 'servidor_whatsapp' from '/content/drive/MyDrive/Dados/notebook_caminho_dos_direitos/servidor_whatsapp.py'>

## 3. Teste sem a Meta (sempre em demonstração)
Simula exatamente o que a Meta manda, com assinatura. Confere: verificação, POST sem assinatura (401), lembrete, toque no botão (o áudio toca aqui), mensagem repetida, PARAR e foto.

In [52]:
from fastapi.testclient import TestClient
from IPython.display import Audio, display

teste = sw.Bot(banco=sw.Banco(':memory:'), pasta_audio='audios_teste')
teste.demonstracao = True                                   # nunca envia nada, mesmo com token
teste.verify_token, teste.app_secret = 'token-de-teste', 'segredo-de-teste'
teste.phone_id = teste.phone_id or '000'
cli = TestClient(sw.criar_app(teste))
TEL = '(11) 98765-4321'

r = cli.get('/webhook', params={'hub.mode': 'subscribe', 'hub.verify_token': 'token-de-teste', 'hub.challenge': '123'})
print('1. verificação:', r.status_code, r.text)
print('2. POST sem assinatura:', cli.post('/webhook', content=b'{}').status_code, '(esperado 401)')

pedido = {'telefone': TEL, 'aceitou_em': '2026-11-14', 'id_encaminhamento': '3729-E1', 'nome': 'Dona Cida',
          'local': 'Poupatempo Santo Amaro', 'data': '18/11', 'hora': '9h',
          'documentos': 'RG, certidão de nascimento e comprovante de endereço', 'formato': 'texto_e_audio'}
print('3. app pede o lembrete:', cli.post('/lembretes', json=pedido).json())
print('   texto que chega:', sw.Lembrete(**{k: v for k, v in pedido.items() if k not in ('telefone', 'aceitou_em')}).texto)

def a_pessoa_manda(**k):
    corpo, ass = sw.evento_simulado(teste, TEL, **k)
    cli.post('/webhook', content=corpo, headers={'X-Hub-Signature-256': ass})
    return corpo, ass

corpo, ass = a_pessoa_manda(botao='AUDIO:3729-E1')
ultimo = teste.saida_demo[-1]
print('4. tocou em "Ouvir em áudio" ->', ultimo['tipo'], '| motor da voz:', voz.falar.ultimo_motor)
display(Audio(ultimo['arquivo']))
n = len(teste.saida_demo)
cli.post('/webhook', content=corpo, headers={'X-Hub-Signature-256': ass})
print('5. mesmo evento de novo -> respostas novas:', len(teste.saida_demo) - n, '(esperado 0)')
a_pessoa_manda(texto='PARAR')
print('6. PARAR ->', teste.saida_demo[-1]['json']['text']['body'])
print('   novo lembrete depois de PARAR:', cli.post('/lembretes', json=pedido).json())
a_pessoa_manda(texto='VOLTAR'); a_pessoa_manda(tipo='image')
print('7. foto ->', teste.saida_demo[-1]['json']['text']['body'])

1. verificação: 200 123
2. POST sem assinatura: 401 (esperado 401)
3. app pede o lembrete: {'enviado': True, 'id_mensagem': 'wamid.DEMO1', 'demonstracao': True}
   texto que chega: Olá, Dona Cida! Lembrete do seu atendimento: Poupatempo Santo Amaro, dia 18/11, às 9h. Leve: RG, certidão de nascimento e comprovante de endereço. Se não puder ir, avise o seu agente. Para não receber mais, responda PARAR.
4. tocou em "Ouvir em áudio" -> audio | motor da voz: neural (edge-tts)


5. mesmo evento de novo -> respostas novas: 0 (esperado 0)
6. PARAR -> Pronto: você não vai mais receber lembretes por este número. Se quiser voltar a receber, é só mandar VOLTAR.
   novo lembrete depois de PARAR: {'detail': 'a pessoa pediu para parar (PARAR)'}
7. foto -> Ainda não recebo áudios, fotos ou arquivos. Para ouvir o lembrete, mande OUVIR. Para outras dúvidas, fale com o seu agente ou com o CRAS.


## 4. Subir o servidor de verdade e criar o endereço público
O Colab não tem endereço público; o **Cloudflare Tunnel** cria um `https://….trycloudflare.com` grátis, sem conta (o manual cita ngrok ou Cloudflare Tunnel). O endereço muda toda vez que rodar: atualize no painel da Meta.

In [53]:
import threading, subprocess, re, time, uvicorn

app = sw.criar_app()            # usa os segredos da seção 2; banco em whatsapp.db
bot = app.state.bot
print('Modo demonstração:', bot.demonstracao)
servidor = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning'))
threading.Thread(target=servidor.run, daemon=True).start()

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
log_tunel = open('tunel.log', 'w')
tunel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                         stdout=log_tunel, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    achou = re.search(r'https://[-\w]+\.trycloudflare\.com', open('tunel.log').read())
    if achou:
        URL = achou.group(0); break
assert URL, 'O túnel não abriu: veja tunel.log'
print(f'''
Painel do app na Meta → WhatsApp → Configuração → Webhook → Editar
  URL de callback:  {URL}/webhook
  Verify token:     (o mesmo WA_VERIFY_TOKEN dos Secrets)
Clique em Verificar e salvar e assine o campo "messages".''')

Modo demonstração: False


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use



Painel do app na Meta → WhatsApp → Configuração → Webhook → Editar
  URL de callback:  https://SEU-ENDERECO.trycloudflare.com/webhook
  Verify token:     (o mesmo WA_VERIFY_TOKEN dos Secrets)
Clique em Verificar e salvar e assine o campo "messages".


## 5. Cadastrar a mensagem-modelo (uma vez só)
Gerenciador do WhatsApp → **Modelos de mensagem** → Criar:
- **Nome:** `lembrete_encaminhamento` · **Categoria:** Utilidade · **Idioma:** Português (BR)
- **Corpo:** o texto abaixo, com as 5 variáveis
- **Botão:** resposta rápida → `Ouvir em áudio`

Enquanto a Meta não aprova, use o **plano B** da seção 6: mande um "oi" do seu celular para o número de teste (abre a janela de 24 h) e o servidor manda o lembrete como texto + áudio sem precisar de modelo.

In [54]:
print(sw.TEMPLATE_TEXTO)

Olá, {{1}}! Lembrete do seu atendimento: {{2}}, dia {{3}}, às {{4}}. Leve: {{5}}. Se não puder ir, avise o seu agente. Para não receber mais, responda PARAR.


## 6. Mandar o lembrete para o seu celular
O número precisa estar cadastrado como destinatário no painel (o número de teste envia para até 5).

In [55]:
MEU_CELULAR = '(11) 9XXXX-XXXX'      # <- seu número, cadastrado no painel da Meta
lembrete = sw.Lembrete('3729-E1', 'Dona Cida', 'Poupatempo Santo Amaro', '18/11', '9h',
                       'RG, certidão de nascimento e comprovante de endereço', formato='texto_e_audio')
PLANO_B = True                       # True = modelo ainda não aprovado (mande "oi" para o número antes)
try:
    if PLANO_B:
        bot.enviar_lembrete_na_janela(MEU_CELULAR, lembrete, aceitou_em='2026-11-14')
    else:
        bot.enviar_lembrete(MEU_CELULAR, lembrete, aceitou_em='2026-11-14')
    print('Enviado! Agora toque em "Ouvir em áudio" no celular (ou mande OUVIR).')
except sw.ErroEnvio as e:
    print('NÃO enviado:', e)

Enviado! Agora toque em "Ouvir em áudio" no celular (ou mande OUVIR).


## 7. Acompanhar: envios e status de entrega (enviada → entregue → lida)

In [56]:
import pandas as pd
display(pd.read_sql('SELECT tipo, id_encaminhamento, status, erro, criado_em FROM envio ORDER BY criado_em DESC', bot.banco.con))
display(pd.read_sql('SELECT tipo, conteudo, recebida_em FROM mensagem_recebida ORDER BY recebida_em DESC', bot.banco.con))

,tipo,id_encaminhamento,status,erro,criado_em
0,audio,3729-E1,aceita,None,2026-11-14T06:17:59+00:00
1,text,None,aceita,None,2026-11-14T06:17:56+00:00
2,audio,3729-E1,aceita,None,2026-11-14T06:16:06+00:00
3,text,None,aceita,None,2026-11-14T06:16:02+00:00
4,audio,3729-E1,aceita,None,2026-11-14T06:14:15+00:00
5,text,None,aceita,None,2026-11-14T06:14:10+00:00
6,template,3729-E1,demonstracao,None,2026-11-14T05:28:38+00:00


,tipo,conteudo,recebida_em


## 8. Desligar

In [57]:
servidor.should_exit = True
tunel.terminate()
print('Servidor e túnel desligados.')

Servidor e túnel desligados.
